## Transform and Load


##### Main tasks:
##### 1. Load dimensions
##### 2. Generate surrogate keys
##### 3. Implement Patient SCD Type 2
##### 4. Load fact tables
##### 5. Create readmission analysis
##### ============================================================


##### Load FactEncounter

In [7]:
-- ============================================================
-- FactEncounter Load - Improved Version
--
-- Source:
--   HP_02_LH_Health_Silver.clinical.encounter
--
-- Target:
--   fact.FactEncounter
--
-- Grain:
--   One row per EncounterID
-- ============================================================


-- ============================================================
-- 1. Prepare source once
-- ============================================================

DROP TABLE IF EXISTS #EncounterPrepared;

CREATE TABLE #EncounterPrepared
(
    EncounterID          VARCHAR(100),
    PatientKey           BIGINT,
    DateKey              INT,

    EncounterCode        VARCHAR(100),
    EncounterDescription VARCHAR(1000),

    ReasonCode           VARCHAR(100),
    ReasonDescription    VARCHAR(1000),

    EncounterCount       INT
)
WITH
(
    DISTRIBUTION = ROUND_ROBIN
);


-- ============================================================
-- 2. Deduplicate source and resolve keys
-- ============================================================

WITH EncounterRanked AS
(
    SELECT
        EncounterID,
        EncounterDate,
        PatientID,

        EncounterCode,
        EncounterDescription,

        ReasonCode,
        ReasonDescription,

        ROW_NUMBER() OVER
        (
            PARTITION BY EncounterID
            ORDER BY EncounterDate DESC
        ) AS rn

    FROM
        [HP_02_LH_Health_Silver].[clinical].[encounter]

    WHERE EncounterID IS NOT NULL
),

EncounterSource AS
(
    SELECT
        EncounterID,
        EncounterDate,
        PatientID,

        EncounterCode,
        EncounterDescription,

        ReasonCode,
        ReasonDescription

    FROM EncounterRanked

    WHERE rn = 1
)

INSERT INTO #EncounterPrepared
(
    EncounterID,
    PatientKey,
    DateKey,

    EncounterCode,
    EncounterDescription,

    ReasonCode,
    ReasonDescription,

    EncounterCount
)

SELECT
    e.EncounterID,

    -- ========================================================
    -- PatientKey
    --
    -- Priority:
    -- 1. SCD2 patient version valid at EncounterDate
    -- 2. Current patient version
    -- 3. Latest historical version
    -- ========================================================

    p.PatientKey,

    -- DateKey
    d.DateKey,

    e.EncounterCode,
    e.EncounterDescription,

    e.ReasonCode,
    e.ReasonDescription,

    1 AS EncounterCount

FROM EncounterSource AS e


OUTER APPLY
(
    SELECT TOP 1
        dp.PatientKey

    FROM dim.DimPatient AS dp

    WHERE
        dp.PatientID = e.PatientID

    ORDER BY
        CASE

            WHEN
                e.EncounterDate IS NOT NULL

                AND CAST(
                    e.EncounterDate AS DATETIME2(6)
                ) >= dp.ValidFrom

                AND CAST(
                    e.EncounterDate AS DATETIME2(6)
                ) < dp.ValidTo

            THEN 0

            WHEN dp.IsCurrent = 1
            THEN 1

            ELSE 2

        END,

        dp.ValidFrom DESC

) AS p


LEFT JOIN dim.DimDate AS d

    ON d.FullDate =
       CAST(e.EncounterDate AS DATE);



-- ============================================================
-- 3. Validate prepared source before modifying FactEncounter
-- ============================================================

SELECT
    COUNT_BIG(*) AS PreparedRows,

    SUM(
        CASE
            WHEN PatientKey IS NULL
            THEN CAST(1 AS BIGINT)
            ELSE CAST(0 AS BIGINT)
        END
    ) AS MissingPatientKey,

    SUM(
        CASE
            WHEN DateKey IS NULL
            THEN CAST(1 AS BIGINT)
            ELSE CAST(0 AS BIGINT)
        END
    ) AS MissingDateKey

FROM #EncounterPrepared;



-- ============================================================
-- 4. Load FactEncounter atomically
-- ============================================================

BEGIN TRY

    BEGIN TRANSACTION;


    -- ========================================================
    -- 4A. Update existing encounters
    --
    -- EncounterKey remains unchanged.
    -- Only update if something actually changed.
    -- ========================================================

    UPDATE target

    SET
        target.PatientKey           = source.PatientKey,
        target.DateKey              = source.DateKey,

        target.EncounterCode        = source.EncounterCode,
        target.EncounterDescription = source.EncounterDescription,

        target.ReasonCode           = source.ReasonCode,
        target.ReasonDescription    = source.ReasonDescription,

        target.EncounterCount       = 1

    FROM fact.FactEncounter AS target

    INNER JOIN #EncounterPrepared AS source

        ON target.EncounterID =
           source.EncounterID

    WHERE
           ISNULL(target.PatientKey, -1)
           <> ISNULL(source.PatientKey, -1)

        OR ISNULL(target.DateKey, -1)
           <> ISNULL(source.DateKey, -1)

        OR ISNULL(target.EncounterCode, '')
           <> ISNULL(source.EncounterCode, '')

        OR ISNULL(target.EncounterDescription, '')
           <> ISNULL(source.EncounterDescription, '')

        OR ISNULL(target.ReasonCode, '')
           <> ISNULL(source.ReasonCode, '')

        OR ISNULL(target.ReasonDescription, '')
           <> ISNULL(source.ReasonDescription, '');


    -- ========================================================
    -- 4B. Find current maximum EncounterKey
    -- ========================================================

    DECLARE @MaxEncounterKey BIGINT;

    SELECT
        @MaxEncounterKey =
            ISNULL(MAX(EncounterKey), 0)

    FROM fact.FactEncounter;


    -- ========================================================
    -- 4C. Insert only NEW encounters
    -- ========================================================

    WITH NewEncounters AS
    (
        SELECT
            source.EncounterID,
            source.PatientKey,
            source.DateKey,

            source.EncounterCode,
            source.EncounterDescription,

            source.ReasonCode,
            source.ReasonDescription,

            source.EncounterCount,

            ROW_NUMBER() OVER
            (
                ORDER BY source.EncounterID
            ) AS RowNumber

        FROM #EncounterPrepared AS source

        WHERE NOT EXISTS
        (
            SELECT 1

            FROM fact.FactEncounter AS target

            WHERE
                target.EncounterID =
                source.EncounterID
        )
    )

    INSERT INTO fact.FactEncounter
    (
        EncounterKey,
        EncounterID,

        PatientKey,
        DateKey,

        EncounterCode,
        EncounterDescription,

        ReasonCode,
        ReasonDescription,

        EncounterCount
    )

    SELECT
        CAST(@MaxEncounterKey AS BIGINT)
        +
        CAST(RowNumber AS BIGINT),

        EncounterID,

        PatientKey,
        DateKey,

        EncounterCode,
        EncounterDescription,

        ReasonCode,
        ReasonDescription,

        EncounterCount

    FROM NewEncounters;


    COMMIT TRANSACTION;

END TRY

BEGIN CATCH

    IF @@TRANCOUNT > 0
        ROLLBACK TRANSACTION;

    THROW;

END CATCH;



-- ============================================================
-- 5. Final validation
-- ============================================================

SELECT
    COUNT_BIG(*) AS TotalEncounterFacts,

    COUNT_BIG(DISTINCT EncounterID)
        AS UniqueEncounterIDs,

    SUM(
        CAST(EncounterCount AS BIGINT)
    ) AS TotalEncounterCount

FROM fact.FactEncounter;


-- ============================================================
-- 6. Duplicate check
--
-- Expected: zero rows
-- ============================================================

SELECT
    EncounterID,
    COUNT_BIG(*) AS RowCounts

FROM fact.FactEncounter

GROUP BY EncounterID

HAVING COUNT_BIG(*) > 1;


-- ============================================================
-- 7. Missing key check
-- ============================================================

SELECT
    SUM(
        CASE
            WHEN PatientKey IS NULL
            THEN CAST(1 AS BIGINT)
            ELSE CAST(0 AS BIGINT)
        END
    ) AS MissingPatientKey,

    SUM(
        CASE
            WHEN DateKey IS NULL
            THEN CAST(1 AS BIGINT)
            ELSE CAST(0 AS BIGINT)
        END
    ) AS MissingDateKey

FROM fact.FactEncounter;


DROP TABLE IF EXISTS #EncounterPrepared;

##### Load FactCondition

In [8]:
-- ============================================================
-- FactCondition Load
--
-- Source:
--   HP_02_LH_Health_Silver.clinical.condition
--
-- Target:
--   fact.FactCondition
--
-- Grain:
--   One row per ConditionID
--
-- Lookups:
--   PatientID      -> DimPatient.PatientKey
--   ConditionCode  -> DimDiagnosis.DiagnosisKey
--   StartDate      -> DimDate.DateKey
--   StopDate       -> DimDate.DateKey
-- ============================================================


-- ============================================================
-- 1. Get current maximum ConditionFactKey
-- ============================================================

DECLARE @MaxConditionFactKey BIGINT;

SELECT
    @MaxConditionFactKey =
        ISNULL(MAX(ConditionFactKey), 0)

FROM fact.FactCondition;


-- ============================================================
-- 2. Prepare source and resolve dimension keys
-- ============================================================

WITH ConditionSource AS
(
    SELECT
        ConditionID,
        PatientID,
        ConditionCode,
        StartDate,
        StopDate

    FROM [HP_02_LH_Health_Silver].[clinical].[condition]

    WHERE ConditionID IS NOT NULL
),

ConditionPrepared AS
(
    SELECT
        c.ConditionID,


        -- ====================================================
        -- PatientKey
        --
        -- Priority:
        -- 1. Patient SCD2 version valid at StartDate
        -- 2. Current patient version
        -- 3. Latest historical patient version
        -- ====================================================

        p.PatientKey,


        -- ====================================================
        -- DiagnosisKey
        -- ====================================================

        d.DiagnosisKey,


        -- ====================================================
        -- Date Keys
        -- ====================================================

        startDate.DateKey AS StartDateKey,

        stopDate.DateKey AS StopDateKey,


        -- Each row represents one condition
        1 AS ConditionCount


    FROM ConditionSource AS c


    -- ========================================================
    -- Patient SCD Type 2 lookup
    -- ========================================================

    OUTER APPLY
    (
        SELECT TOP 1
            dp.PatientKey

        FROM dim.DimPatient AS dp

        WHERE
            dp.PatientID = c.PatientID

        ORDER BY
            CASE

                WHEN
                    c.StartDate IS NOT NULL

                    AND CAST(
                        c.StartDate
                        AS DATETIME2(6)
                    ) >= dp.ValidFrom

                    AND CAST(
                        c.StartDate
                        AS DATETIME2(6)
                    ) < dp.ValidTo

                THEN 0


                WHEN dp.IsCurrent = 1
                THEN 1


                ELSE 2

            END,

            dp.ValidFrom DESC

    ) AS p


    -- ========================================================
    -- Diagnosis lookup
    -- ========================================================

    LEFT JOIN dim.DimDiagnosis AS d

        ON d.DiagnosisCode =
           c.ConditionCode


    -- ========================================================
    -- Start Date lookup
    -- ========================================================

    LEFT JOIN dim.DimDate AS startDate

        ON startDate.FullDate =
           CAST(c.StartDate AS DATE)


    -- ========================================================
    -- Stop Date lookup
    --
    -- NULL StopDate is valid for active conditions.
    -- ========================================================

    LEFT JOIN dim.DimDate AS stopDate

        ON stopDate.FullDate =
           CAST(c.StopDate AS DATE)
),


-- ============================================================
-- 3. Find only new ConditionIDs
-- ============================================================

NewConditions AS
(
    SELECT
        source.ConditionID,

        source.PatientKey,
        source.DiagnosisKey,

        source.StartDateKey,
        source.StopDateKey,

        source.ConditionCount,

        ROW_NUMBER() OVER
        (
            ORDER BY source.ConditionID
        ) AS RowNumber

    FROM ConditionPrepared AS source

    WHERE NOT EXISTS
    (
        SELECT 1

        FROM fact.FactCondition AS target

        WHERE
            target.ConditionID =
            source.ConditionID
    )
)


-- ============================================================
-- 4. Insert new FactCondition rows
-- ============================================================

INSERT INTO fact.FactCondition
(
    ConditionFactKey,
    ConditionID,

    PatientKey,
    DiagnosisKey,

    StartDateKey,
    StopDateKey,

    ConditionCount
)

SELECT
    CAST(@MaxConditionFactKey AS BIGINT)
    +
    CAST(RowNumber AS BIGINT),

    ConditionID,

    PatientKey,
    DiagnosisKey,

    StartDateKey,
    StopDateKey,

    ConditionCount

FROM NewConditions;


-- ============================================================
-- 5. Update existing FactCondition rows
--
-- Allows corrected source values or late-arriving
-- dimensions to populate previously missing keys.
-- ============================================================

WITH ConditionSource AS
(
    SELECT
        ConditionID,
        PatientID,
        ConditionCode,
        StartDate,
        StopDate

    FROM [HP_02_LH_Health_Silver].[clinical].[condition]

    WHERE ConditionID IS NOT NULL
),

ConditionPrepared AS
(
    SELECT
        c.ConditionID,

        p.PatientKey,

        d.DiagnosisKey,

        startDate.DateKey AS StartDateKey,
        stopDate.DateKey AS StopDateKey

    FROM ConditionSource AS c


    -- ========================================================
    -- Patient SCD2 lookup
    -- ========================================================

    OUTER APPLY
    (
        SELECT TOP 1
            dp.PatientKey

        FROM dim.DimPatient AS dp

        WHERE
            dp.PatientID =
            c.PatientID

        ORDER BY
            CASE

                WHEN
                    c.StartDate IS NOT NULL

                    AND CAST(
                        c.StartDate
                        AS DATETIME2(6)
                    ) >= dp.ValidFrom

                    AND CAST(
                        c.StartDate
                        AS DATETIME2(6)
                    ) < dp.ValidTo

                THEN 0


                WHEN dp.IsCurrent = 1
                THEN 1


                ELSE 2

            END,

            dp.ValidFrom DESC

    ) AS p


    -- ========================================================
    -- Diagnosis lookup
    -- ========================================================

    LEFT JOIN dim.DimDiagnosis AS d

        ON d.DiagnosisCode =
           c.ConditionCode


    -- ========================================================
    -- Start Date
    -- ========================================================

    LEFT JOIN dim.DimDate AS startDate

        ON startDate.FullDate =
           CAST(c.StartDate AS DATE)


    -- ========================================================
    -- Stop Date
    -- ========================================================

    LEFT JOIN dim.DimDate AS stopDate

        ON stopDate.FullDate =
           CAST(c.StopDate AS DATE)
)


UPDATE target

SET
    target.PatientKey =
        source.PatientKey,

    target.DiagnosisKey =
        source.DiagnosisKey,

    target.StartDateKey =
        source.StartDateKey,

    target.StopDateKey =
        source.StopDateKey,

    target.ConditionCount = 1

FROM fact.FactCondition AS target

INNER JOIN ConditionPrepared AS source

    ON target.ConditionID =
       source.ConditionID;


-- ============================================================
-- 6. Validate missing dimension keys
-- ============================================================

SELECT
    COUNT_BIG(*) AS TotalConditions,

    SUM(
        CASE
            WHEN PatientKey IS NULL
            THEN CAST(1 AS BIGINT)
            ELSE CAST(0 AS BIGINT)
        END
    ) AS MissingPatientKey,

    SUM(
        CASE
            WHEN DiagnosisKey IS NULL
            THEN CAST(1 AS BIGINT)
            ELSE CAST(0 AS BIGINT)
        END
    ) AS MissingDiagnosisKey,

    SUM(
        CASE
            WHEN StartDateKey IS NULL
            THEN CAST(1 AS BIGINT)
            ELSE CAST(0 AS BIGINT)
        END
    ) AS MissingStartDateKey

FROM fact.FactCondition;


-- ============================================================
-- 7. Check duplicate ConditionIDs
--
-- Expected:
--   zero rows
-- ============================================================

SELECT
    ConditionID,
    COUNT_BIG(*) AS RowCounts

FROM fact.FactCondition

GROUP BY ConditionID

HAVING COUNT_BIG(*) > 1;

##### Load FactMedication

In [10]:
-- ============================================================
-- FactMedication Load
--
-- Source:
--   HP_02_LH_Health_Silver.clinical.medication
--
-- Target:
--   fact.FactMedication
--
-- Grain:
--   One row per MedicationID
--
-- Lookups:
--   PatientID      -> DimPatient.PatientKey
--   MedicationCode -> DimMedication.MedicationKey
--   StartDate      -> DimDate.DateKey
--   StopDate       -> DimDate.DateKey
-- ============================================================


-- ============================================================
-- 1. Get maximum MedicationFactKey
-- ============================================================

DECLARE @MaxMedicationFactKey BIGINT;

SELECT
    @MaxMedicationFactKey =
        ISNULL(MAX(MedicationFactKey), 0)

FROM fact.FactMedication;


-- ============================================================
-- 2. Prepare source and resolve dimension keys
-- ============================================================

WITH MedicationSource AS
(
    SELECT
        MedicationID,
        PatientID,
        MedicationCode,
        StartDate,
        StopDate

    FROM [HP_02_LH_Health_Silver].[clinical].[medication]

    WHERE MedicationID IS NOT NULL
),

MedicationPrepared AS
(
    SELECT
        m.MedicationID,


        -- ====================================================
        -- PatientKey
        --
        -- Priority:
        -- 1. Patient SCD2 version valid at StartDate
        -- 2. Current patient version
        -- 3. Latest historical patient version
        -- ====================================================

        p.PatientKey,


        -- ====================================================
        -- MedicationKey
        -- ====================================================

        md.MedicationKey,


        -- ====================================================
        -- Date Keys
        -- ====================================================

        startDate.DateKey AS StartDateKey,

        stopDate.DateKey AS StopDateKey,


        -- One row represents one medication event
        1 AS MedicationCount


    FROM MedicationSource AS m


    -- ========================================================
    -- Patient SCD Type 2 lookup
    -- ========================================================

    OUTER APPLY
    (
        SELECT TOP 1
            dp.PatientKey

        FROM dim.DimPatient AS dp

        WHERE
            dp.PatientID = m.PatientID

        ORDER BY
            CASE

                -- Patient version valid when medication started
                WHEN
                    m.StartDate IS NOT NULL

                    AND CAST(
                        m.StartDate
                        AS DATETIME2(6)
                    ) >= dp.ValidFrom

                    AND CAST(
                        m.StartDate
                        AS DATETIME2(6)
                    ) < dp.ValidTo

                THEN 0


                -- Current patient version
                WHEN dp.IsCurrent = 1
                THEN 1


                -- Latest historical version
                ELSE 2

            END,

            dp.ValidFrom DESC

    ) AS p


    -- ========================================================
    -- Medication dimension lookup
    -- ========================================================

    LEFT JOIN dim.DimMedication AS md

        ON md.MedicationCode =
           m.MedicationCode


    -- ========================================================
    -- Start Date lookup
    -- ========================================================

    LEFT JOIN dim.DimDate AS startDate

        ON startDate.FullDate =
           CAST(m.StartDate AS DATE)


    -- ========================================================
    -- Stop Date lookup
    --
    -- NULL StopDate is valid for an active medication.
    -- ========================================================

    LEFT JOIN dim.DimDate AS stopDate

        ON stopDate.FullDate =
           CAST(m.StopDate AS DATE)
),


-- ============================================================
-- 3. Find only NEW MedicationIDs
--
-- Existing MedicationFactKey remains unchanged.
-- ============================================================

NewMedications AS
(
    SELECT
        source.MedicationID,

        source.PatientKey,
        source.MedicationKey,

        source.StartDateKey,
        source.StopDateKey,

        source.MedicationCount,

        ROW_NUMBER() OVER
        (
            ORDER BY source.MedicationID
        ) AS RowNumber

    FROM MedicationPrepared AS source

    WHERE NOT EXISTS
    (
        SELECT 1

        FROM fact.FactMedication AS target

        WHERE
            target.MedicationID =
            source.MedicationID
    )
)


-- ============================================================
-- 4. Insert new medication facts
-- ============================================================

INSERT INTO fact.FactMedication
(
    MedicationFactKey,
    MedicationID,

    PatientKey,
    MedicationKey,

    StartDateKey,
    StopDateKey,

    MedicationCount
)

SELECT
    CAST(@MaxMedicationFactKey AS BIGINT)
    +
    CAST(RowNumber AS BIGINT),

    MedicationID,

    PatientKey,
    MedicationKey,

    StartDateKey,
    StopDateKey,

    MedicationCount

FROM NewMedications;


-- ============================================================
-- 5. Update existing medication facts
--
-- Useful for:
--   late-arriving dimensions
--   corrected dates
--   previously missing dimension keys
-- ============================================================

WITH MedicationSource AS
(
    SELECT
        MedicationID,
        PatientID,
        MedicationCode,
        StartDate,
        StopDate

    FROM [HP_02_LH_Health_Silver].[clinical].[medication]

    WHERE MedicationID IS NOT NULL
),

MedicationPrepared AS
(
    SELECT
        m.MedicationID,

        p.PatientKey,

        md.MedicationKey,

        startDate.DateKey AS StartDateKey,

        stopDate.DateKey AS StopDateKey


    FROM MedicationSource AS m


    -- ========================================================
    -- Patient SCD2 lookup
    -- ========================================================

    OUTER APPLY
    (
        SELECT TOP 1
            dp.PatientKey

        FROM dim.DimPatient AS dp

        WHERE
            dp.PatientID =
            m.PatientID

        ORDER BY
            CASE

                WHEN
                    m.StartDate IS NOT NULL

                    AND CAST(
                        m.StartDate
                        AS DATETIME2(6)
                    ) >= dp.ValidFrom

                    AND CAST(
                        m.StartDate
                        AS DATETIME2(6)
                    ) < dp.ValidTo

                THEN 0


                WHEN dp.IsCurrent = 1
                THEN 1


                ELSE 2

            END,

            dp.ValidFrom DESC

    ) AS p


    -- ========================================================
    -- Medication lookup
    -- ========================================================

    LEFT JOIN dim.DimMedication AS md

        ON md.MedicationCode =
           m.MedicationCode


    -- ========================================================
    -- Start Date lookup
    -- ========================================================

    LEFT JOIN dim.DimDate AS startDate

        ON startDate.FullDate =
           CAST(m.StartDate AS DATE)


    -- ========================================================
    -- Stop Date lookup
    -- ========================================================

    LEFT JOIN dim.DimDate AS stopDate

        ON stopDate.FullDate =
           CAST(m.StopDate AS DATE)
)


UPDATE target

SET
    target.PatientKey =
        source.PatientKey,

    target.MedicationKey =
        source.MedicationKey,

    target.StartDateKey =
        source.StartDateKey,

    target.StopDateKey =
        source.StopDateKey,

    target.MedicationCount = 1

FROM fact.FactMedication AS target

INNER JOIN MedicationPrepared AS source

    ON target.MedicationID =
       source.MedicationID;



##### Load FactProcedure

In [12]:
-- ============================================================
-- FactProcedure Load
--
-- Source:
--   HP_02_LH_Health_Silver.clinical.procedure
--
-- Target:
--   fact.FactProcedure
--
-- Grain:
--   One row per ProcedureID
--
-- Lookups:
--   PatientID      -> DimPatient.PatientKey
--   ProcedureCode  -> DimProcedure.ProcedureKey
--   ProcedureDate  -> DimDate.DateKey
-- ============================================================


-- ============================================================
-- 1. Get current maximum ProcedureFactKey
-- ============================================================

DECLARE @MaxProcedureFactKey BIGINT;

SELECT
    @MaxProcedureFactKey =
        ISNULL(MAX(ProcedureFactKey), 0)

FROM fact.FactProcedure;


-- ============================================================
-- 2. Prepare source and resolve surrogate keys
-- ============================================================

WITH ProcedureSource AS
(
    SELECT
        ProcedureID,
        PatientID,
        ProcedureCode,
        ProcedureDate

    FROM [HP_02_LH_Health_Silver].[clinical].[procedure]

    WHERE ProcedureID IS NOT NULL
),

ProcedurePrepared AS
(
    SELECT
        pr.ProcedureID,


        -- ====================================================
        -- PatientKey
        --
        -- Priority:
        -- 1. Patient version valid at ProcedureDate
        -- 2. Current patient version
        -- 3. Latest historical patient version
        -- ====================================================

        p.PatientKey,


        -- ====================================================
        -- ProcedureKey
        -- ====================================================

        dp.ProcedureKey,


        -- ====================================================
        -- Procedure Date
        -- ====================================================

        dd.DateKey AS ProcedureDateKey,


        -- One row = one procedure
        1 AS ProcedureCount


    FROM ProcedureSource AS pr


    -- ========================================================
    -- Patient SCD Type 2 lookup
    -- ========================================================

    OUTER APPLY
    (
        SELECT TOP 1
            patient.PatientKey

        FROM dim.DimPatient AS patient

        WHERE
            patient.PatientID = pr.PatientID

        ORDER BY
            CASE

                WHEN
                    pr.ProcedureDate IS NOT NULL

                    AND CAST(
                        pr.ProcedureDate
                        AS DATETIME2(6)
                    ) >= patient.ValidFrom

                    AND CAST(
                        pr.ProcedureDate
                        AS DATETIME2(6)
                    ) < patient.ValidTo

                THEN 0


                WHEN patient.IsCurrent = 1
                THEN 1


                ELSE 2

            END,

            patient.ValidFrom DESC

    ) AS p


    -- ========================================================
    -- Procedure dimension lookup
    -- ========================================================

    LEFT JOIN dim.DimProcedure AS dp

        ON dp.ProcedureCode =
           pr.ProcedureCode


    -- ========================================================
    -- Date dimension lookup
    -- ========================================================

    LEFT JOIN dim.DimDate AS dd

        ON dd.FullDate =
           CAST(pr.ProcedureDate AS DATE)
),


-- ============================================================
-- 3. Find only NEW ProcedureIDs
--
-- Existing ProcedureFactKey values remain unchanged.
-- ============================================================

NewProcedures AS
(
    SELECT
        source.ProcedureID,

        source.PatientKey,
        source.ProcedureKey,

        source.ProcedureDateKey,

        source.ProcedureCount,

        ROW_NUMBER() OVER
        (
            ORDER BY source.ProcedureID
        ) AS RowNumber

    FROM ProcedurePrepared AS source

    WHERE NOT EXISTS
    (
        SELECT 1

        FROM fact.FactProcedure AS target

        WHERE
            target.ProcedureID =
            source.ProcedureID
    )
)


-- ============================================================
-- 4. Insert new FactProcedure rows
-- ============================================================

INSERT INTO fact.FactProcedure
(
    ProcedureFactKey,
    ProcedureID,

    PatientKey,
    ProcedureKey,

    ProcedureDateKey,

    ProcedureCount
)

SELECT
    CAST(@MaxProcedureFactKey AS BIGINT)
    +
    CAST(RowNumber AS BIGINT),

    ProcedureID,

    PatientKey,
    ProcedureKey,

    ProcedureDateKey,

    ProcedureCount

FROM NewProcedures;


-- ============================================================
-- 5. Update existing Procedure facts
--
-- Allows corrected / late-arriving dimension keys
-- without changing ProcedureFactKey.
-- ============================================================

WITH ProcedureSource AS
(
    SELECT
        ProcedureID,
        PatientID,
        ProcedureCode,
        ProcedureDate

    FROM [HP_02_LH_Health_Silver].[clinical].[procedure]

    WHERE ProcedureID IS NOT NULL
),

ProcedurePrepared AS
(
    SELECT
        pr.ProcedureID,

        p.PatientKey,

        dp.ProcedureKey,

        dd.DateKey AS ProcedureDateKey


    FROM ProcedureSource AS pr


    -- ========================================================
    -- Patient SCD2 lookup
    -- ========================================================

    OUTER APPLY
    (
        SELECT TOP 1
            patient.PatientKey

        FROM dim.DimPatient AS patient

        WHERE
            patient.PatientID =
            pr.PatientID

        ORDER BY
            CASE

                WHEN
                    pr.ProcedureDate IS NOT NULL

                    AND CAST(
                        pr.ProcedureDate
                        AS DATETIME2(6)
                    ) >= patient.ValidFrom

                    AND CAST(
                        pr.ProcedureDate
                        AS DATETIME2(6)
                    ) < patient.ValidTo

                THEN 0


                WHEN patient.IsCurrent = 1
                THEN 1


                ELSE 2

            END,

            patient.ValidFrom DESC

    ) AS p


    -- ========================================================
    -- Procedure lookup
    -- ========================================================

    LEFT JOIN dim.DimProcedure AS dp

        ON dp.ProcedureCode =
           pr.ProcedureCode


    -- ========================================================
    -- Procedure Date
    -- ========================================================

    LEFT JOIN dim.DimDate AS dd

        ON dd.FullDate =
           CAST(pr.ProcedureDate AS DATE)
)


UPDATE target

SET
    target.PatientKey =
        source.PatientKey,

    target.ProcedureKey =
        source.ProcedureKey,

    target.ProcedureDateKey =
        source.ProcedureDateKey,

    target.ProcedureCount = 1

FROM fact.FactProcedure AS target

INNER JOIN ProcedurePrepared AS source

    ON target.ProcedureID =
       source.ProcedureID;


##### Load FactObservation

In [15]:
-- ============================================================
-- FactObservation Load
--
-- Source:
--   HP_02_LH_Health_Silver.clinical.observation
--
-- Target:
--   fact.FactObservation
--
-- Grain:
--   One row per ObservationID
--
-- Lookups:
--   PatientID       -> DimPatient.PatientKey
--   ObservationDate -> DimDate.DateKey
--
-- Measures / attributes:
--   ObservationCode
--   ObservationDescription
--   ObservationValue
--   NumericValue
--   Units
--   ObservationCount = 1
-- ============================================================


-- ============================================================
-- 1. Get current maximum ObservationFactKey
-- ============================================================

DECLARE @MaxObservationFactKey BIGINT;

SELECT
    @MaxObservationFactKey =
        ISNULL(MAX(ObservationFactKey), 0)

FROM fact.FactObservation;


-- ============================================================
-- 2. Prepare source and resolve dimension keys
-- ============================================================

WITH ObservationSource AS
(
    SELECT
        ObservationID,
        PatientID,
        ObservationDate,

        ObservationCode,
        ObservationDescription,

        ObservationValue,
        NumericValue,
        Units

    FROM [HP_02_LH_Health_Silver].[clinical].[observation]

    WHERE ObservationID IS NOT NULL
),

ObservationPrepared AS
(
    SELECT
        o.ObservationID,


        -- ====================================================
        -- PatientKey
        --
        -- Priority:
        -- 1. Patient SCD2 version valid at ObservationDate
        -- 2. Current patient version
        -- 3. Latest historical patient version
        -- ====================================================

        p.PatientKey,


        -- ====================================================
        -- DateKey
        -- ====================================================

        d.DateKey,


        -- ====================================================
        -- Observation attributes
        -- ====================================================

        o.ObservationCode,
        o.ObservationDescription,

        o.ObservationValue,
        o.NumericValue,

        o.Units,


        -- One source row = one observation
        1 AS ObservationCount


    FROM ObservationSource AS o


    -- ========================================================
    -- Patient SCD Type 2 lookup
    -- ========================================================

    OUTER APPLY
    (
        SELECT TOP 1
            dp.PatientKey

        FROM dim.DimPatient AS dp

        WHERE
            dp.PatientID = o.PatientID

        ORDER BY
            CASE

                WHEN
                    o.ObservationDate IS NOT NULL

                    AND CAST(
                        o.ObservationDate
                        AS DATETIME2(6)
                    ) >= dp.ValidFrom

                    AND CAST(
                        o.ObservationDate
                        AS DATETIME2(6)
                    ) < dp.ValidTo

                THEN 0


                WHEN dp.IsCurrent = 1
                THEN 1


                ELSE 2

            END,

            dp.ValidFrom DESC

    ) AS p


    -- ========================================================
    -- Date dimension lookup
    -- ========================================================

    LEFT JOIN dim.DimDate AS d

        ON d.FullDate =
           CAST(o.ObservationDate AS DATE)
),


-- ============================================================
-- 3. Find only NEW ObservationIDs
--
-- Existing ObservationFactKey remains unchanged.
-- ============================================================

NewObservations AS
(
    SELECT
        source.ObservationID,

        source.PatientKey,
        source.DateKey,

        source.ObservationCode,
        source.ObservationDescription,

        source.ObservationValue,
        source.NumericValue,

        source.Units,

        source.ObservationCount,

        ROW_NUMBER() OVER
        (
            ORDER BY source.ObservationID
        ) AS RowNumber

    FROM ObservationPrepared AS source

    WHERE NOT EXISTS
    (
        SELECT 1

        FROM fact.FactObservation AS target

        WHERE
            target.ObservationID =
            source.ObservationID
    )
)


-- ============================================================
-- 4. Insert new observations
-- ============================================================

INSERT INTO fact.FactObservation
(
    ObservationFactKey,
    ObservationID,

    PatientKey,
    DateKey,

    ObservationCode,
    ObservationDescription,

    ObservationValue,
    NumericValue,

    Units,

    ObservationCount
)

SELECT
    CAST(@MaxObservationFactKey AS BIGINT)
    +
    CAST(RowNumber AS BIGINT),

    ObservationID,

    PatientKey,
    DateKey,

    ObservationCode,
    ObservationDescription,

    ObservationValue,
    NumericValue,

    Units,

    ObservationCount

FROM NewObservations;


-- ============================================================
-- 5. Update existing FactObservation rows
--
-- Handles:
--   corrected observations
--   late-arriving dimensions
--   previously missing dimension keys
--
-- ObservationFactKey remains unchanged.
-- ============================================================

WITH ObservationSource AS
(
    SELECT
        ObservationID,
        PatientID,
        ObservationDate,

        ObservationCode,
        ObservationDescription,

        ObservationValue,
        NumericValue,
        Units

    FROM [HP_02_LH_Health_Silver].[clinical].[observation]

    WHERE ObservationID IS NOT NULL
),

ObservationPrepared AS
(
    SELECT
        o.ObservationID,

        p.PatientKey,

        d.DateKey,

        o.ObservationCode,
        o.ObservationDescription,

        o.ObservationValue,
       CAST(
            o.NumericValue
            AS DECIMAL(18,4)
        ) AS NumericValue,

        o.Units


    FROM ObservationSource AS o


    -- ========================================================
    -- Patient SCD2 lookup
    -- ========================================================

    OUTER APPLY
    (
        SELECT TOP 1
            dp.PatientKey

        FROM dim.DimPatient AS dp

        WHERE
            dp.PatientID =
            o.PatientID

        ORDER BY
            CASE

                WHEN
                    o.ObservationDate IS NOT NULL

                    AND CAST(
                        o.ObservationDate
                        AS DATETIME2(6)
                    ) >= dp.ValidFrom

                    AND CAST(
                        o.ObservationDate
                        AS DATETIME2(6)
                    ) < dp.ValidTo

                THEN 0


                WHEN dp.IsCurrent = 1
                THEN 1


                ELSE 2

            END,

            dp.ValidFrom DESC

    ) AS p


    -- ========================================================
    -- Date lookup
    -- ========================================================

    LEFT JOIN dim.DimDate AS d

        ON d.FullDate =
           CAST(o.ObservationDate AS DATE)
)


UPDATE target

SET
    target.PatientKey =
        source.PatientKey,

    target.DateKey =
        source.DateKey,

    target.ObservationCode =
        source.ObservationCode,

    target.ObservationDescription =
        source.ObservationDescription,

    target.ObservationValue =
        source.ObservationValue,

    target.NumericValue =
        source.NumericValue,

    target.Units =
        source.Units,

    target.ObservationCount = 1

FROM fact.FactObservation AS target

INNER JOIN ObservationPrepared AS source

    ON target.ObservationID =
       source.ObservationID;




##### FactReadmission

In [14]:
-- ============================================================
-- FactReadmission
--
-- TEMPORARY / PROXY METHOD
--
-- Current source does not have:
--   AdmissionDate
--   DischargeDate
--   EncounterClass
--
-- Therefore EncounterDate is used as a proxy.
--
-- Grain:
--   One row per index encounter that has a later encounter
-- ============================================================


-- ============================================================
-- 1. Find current maximum ReadmissionKey
-- ============================================================

DECLARE @MaxReadmissionKey BIGINT;

SELECT
    @MaxReadmissionKey =
        ISNULL(MAX(ReadmissionKey), 0)

FROM fact.FactReadmission;


-- ============================================================
-- 2. Order encounters for each patient
--
-- LEAD() finds the patient's next encounter
-- ============================================================

WITH EncounterSequence AS
(
    SELECT
        e.PatientKey,

        e.EncounterKey
            AS IndexEncounterKey,
        d.DateKey,

        d.FullDate
            AS DischargeDate,

        LEAD(e.EncounterKey) OVER
        (
            PARTITION BY e.PatientKey
            ORDER BY
                d.FullDate,
                e.EncounterKey
        ) AS ReadmissionEncounterKey,

        LEAD(d.FullDate) OVER
        (
            PARTITION BY e.PatientKey
            ORDER BY
                d.FullDate,
                e.EncounterKey
        ) AS ReadmissionDate

    FROM fact.FactEncounter AS e

    INNER JOIN dim.DimDate AS d
        ON d.DateKey = e.DateKey

    WHERE
        e.PatientKey IS NOT NULL
        AND e.DateKey IS NOT NULL
),


-- ============================================================
-- 3. Calculate days until next encounter
-- ============================================================

ReadmissionSource AS
(
    SELECT
        PatientKey,
        IndexEncounterKey,
        ReadmissionEncounterKey,
        DateKey,
        DischargeDate,
        ReadmissionDate,

        DATEDIFF(
            DAY,
            DischargeDate,
            ReadmissionDate
        ) AS DaysToReadmission

    FROM EncounterSequence

    WHERE
        ReadmissionEncounterKey IS NOT NULL
),


-- ============================================================
-- 4. Keep encounters with a later encounter
--
-- We exclude same-day encounters.
-- ============================================================

ReadmissionPrepared AS
(
    SELECT
        PatientKey,
        IndexEncounterKey,
        ReadmissionEncounterKey,
        DateKey,
        DischargeDate,
        ReadmissionDate,

        DaysToReadmission,

        CAST(
            CASE
                WHEN DaysToReadmission
                     BETWEEN 1 AND 30
                THEN 1
                ELSE 0
            END
            AS BIT
        ) AS IsReadmission30Day

    FROM ReadmissionSource

    WHERE DaysToReadmission > 0
),


-- ============================================================
-- 5. Insert only new index/readmission pairs
-- ============================================================

NewReadmissions AS
(
    SELECT
        source.*,

        ROW_NUMBER() OVER
        (
            ORDER BY
                source.PatientKey,
                source.DischargeDate,
                source.IndexEncounterKey
        ) AS RowNumber

    FROM ReadmissionPrepared AS source

    WHERE NOT EXISTS
    (
        SELECT 1

        FROM fact.FactReadmission AS target

        WHERE
            target.IndexEncounterKey =
                source.IndexEncounterKey

            AND target.ReadmissionEncounterKey =
                source.ReadmissionEncounterKey
    )
)


INSERT INTO fact.FactReadmission
(
    ReadmissionKey,
    DateKey,
    PatientKey,

    IndexEncounterKey,
    ReadmissionEncounterKey,

    DischargeDate,
    ReadmissionDate,

    DaysToReadmission,

    IsReadmission30Day
)

SELECT
    CAST(@MaxReadmissionKey AS BIGINT)
    +
    CAST(RowNumber AS BIGINT),
    DateKey,
    PatientKey,

    IndexEncounterKey,
    ReadmissionEncounterKey,

    DischargeDate,
    ReadmissionDate,

    DaysToReadmission,

    IsReadmission30Day

FROM NewReadmissions;

##### FactFacilityDaily

In [16]:
-- ============================================================
-- FactFacilityDaily Load
--
-- Grain:
--   One row per FacilityKey + DateKey
--
-- Sources:
--   fact.FactEncounter
--   fact.FactReadmission
--
-- Target:
--   fact.FactFacilityDaily
--
-- Cost definition:
--   TotalEncounterCost   = SUM(BaseEncounterCost)
--   AverageEncounterCost = AVG(BaseEncounterCost)
-- ============================================================


-- ============================================================
-- 1. UPDATE EXISTING FACILITY-DAY ROWS
-- ============================================================

WITH EncounterDaily AS
(
    SELECT
        e.DateKey,
        e.FacilityKey,


        -- ====================================================
        -- Distinct patients seen at the facility that day
        -- ====================================================

        COUNT_BIG(DISTINCT e.PatientKey)
            AS PatientCount,


        -- ====================================================
        -- Total encounters
        -- ====================================================

        COUNT_BIG(*)
            AS EncounterCount,


        -- ====================================================
        -- Emergency visits
        -- ====================================================

        SUM(
            CASE
                WHEN LOWER(e.EncounterClass) = 'emergency'
                THEN CAST(1 AS BIGINT)
                ELSE CAST(0 AS BIGINT)
            END
        ) AS EmergencyVisitCount,


        -- ====================================================
        -- Inpatient visits
        -- ====================================================

        SUM(
            CASE
                WHEN LOWER(e.EncounterClass) = 'inpatient'
                THEN CAST(1 AS BIGINT)
                ELSE CAST(0 AS BIGINT)
            END
        ) AS InpatientVisitCount,


        -- ====================================================
        -- Outpatient / ambulatory visits
        -- ====================================================

        SUM(
            CASE
                WHEN LOWER(e.EncounterClass)
                     IN ('outpatient', 'ambulatory')
                THEN CAST(1 AS BIGINT)
                ELSE CAST(0 AS BIGINT)
            END
        ) AS OutpatientVisitCount,


        -- ====================================================
        -- Average Length of Stay
        -- ====================================================

        AVG(
            CAST(
                e.LengthOfStayDays
                AS FLOAT
            )
        ) AS AverageLengthOfStay,


        -- ====================================================
        -- Total Encounter Cost
        --
        -- Sum of BaseEncounterCost
        -- ====================================================

        SUM(
            CAST(
                ISNULL(e.BaseEncounterCost, 0)
                AS DECIMAL(18,2)
            )
        ) AS TotalEncounterCost,


        -- ====================================================
        -- Average Encounter Cost
        -- ====================================================

        AVG(
            CAST(
                e.BaseEncounterCost
                AS DECIMAL(18,2)
            )
        ) AS AverageEncounterCost


    FROM fact.FactEncounter AS e

    WHERE
        e.DateKey IS NOT NULL
        AND e.FacilityKey IS NOT NULL

    GROUP BY
        e.DateKey,
        e.FacilityKey
),


-- ============================================================
-- 2. Calculate 30-day readmissions
--
-- Readmission is assigned to the facility/date of the
-- readmission encounter.
-- ============================================================

ReadmissionDaily AS
(
    SELECT
        e.DateKey,
        e.FacilityKey,

        COUNT_BIG(*)
            AS ReadmissionCount

    FROM fact.FactReadmission AS r

    INNER JOIN fact.FactEncounter AS e

        ON e.EncounterKey =
           r.ReadmissionEncounterKey

    WHERE
        r.IsReadmission30Day = 1

        AND e.DateKey IS NOT NULL
        AND e.FacilityKey IS NOT NULL

    GROUP BY
        e.DateKey,
        e.FacilityKey
),


-- ============================================================
-- 3. Combine Encounter + Readmission metrics
-- ============================================================

FacilityDailySource AS
(
    SELECT
        e.DateKey,
        e.FacilityKey,

        e.PatientCount,
        e.EncounterCount,

        e.EmergencyVisitCount,
        e.InpatientVisitCount,
        e.OutpatientVisitCount,

        ISNULL(
            r.ReadmissionCount,
            CAST(0 AS BIGINT)
        ) AS ReadmissionCount,

        e.AverageLengthOfStay,

        e.TotalEncounterCost,
        e.AverageEncounterCost

    FROM EncounterDaily AS e

    LEFT JOIN ReadmissionDaily AS r

        ON r.DateKey =
           e.DateKey

        AND r.FacilityKey =
            e.FacilityKey
)


-- ============================================================
-- 4. Update rows already in FactFacilityDaily
-- ============================================================

UPDATE target

SET
    target.PatientCount =
        source.PatientCount,

    target.EncounterCount =
        source.EncounterCount,

    target.EmergencyVisitCount =
        source.EmergencyVisitCount,

    target.InpatientVisitCount =
        source.InpatientVisitCount,

    target.OutpatientVisitCount =
        source.OutpatientVisitCount,

    target.ReadmissionCount =
        source.ReadmissionCount,

    target.AverageLengthOfStay =
        source.AverageLengthOfStay,

    target.TotalEncounterCost =
        source.TotalEncounterCost,

    target.AverageEncounterCost =
        source.AverageEncounterCost


FROM fact.FactFacilityDaily AS target

INNER JOIN FacilityDailySource AS source

    ON target.DateKey =
       source.DateKey

    AND target.FacilityKey =
        source.FacilityKey;



-- ============================================================
-- 5. Get maximum FacilityDailyKey
-- ============================================================

DECLARE @MaxFacilityDailyKey BIGINT;

SELECT
    @MaxFacilityDailyKey =
        ISNULL(MAX(FacilityDailyKey), 0)

FROM fact.FactFacilityDaily;



-- ============================================================
-- 6. Recalculate source for INSERT
-- ============================================================

WITH EncounterDaily AS
(
    SELECT
        e.DateKey,
        e.FacilityKey,


        COUNT_BIG(DISTINCT e.PatientKey)
            AS PatientCount,


        COUNT_BIG(*)
            AS EncounterCount,


        SUM(
            CASE
                WHEN LOWER(e.EncounterClass) = 'emergency'
                THEN CAST(1 AS BIGINT)
                ELSE CAST(0 AS BIGINT)
            END
        ) AS EmergencyVisitCount,


        SUM(
            CASE
                WHEN LOWER(e.EncounterClass) = 'inpatient'
                THEN CAST(1 AS BIGINT)
                ELSE CAST(0 AS BIGINT)
            END
        ) AS InpatientVisitCount,


        SUM(
            CASE
                WHEN LOWER(e.EncounterClass)
                     IN ('outpatient', 'ambulatory')
                THEN CAST(1 AS BIGINT)
                ELSE CAST(0 AS BIGINT)
            END
        ) AS OutpatientVisitCount,


        AVG(
            CAST(
                e.LengthOfStayDays
                AS FLOAT
            )
        ) AS AverageLengthOfStay,


        SUM(
            CAST(
                ISNULL(e.BaseEncounterCost, 0)
                AS DECIMAL(18,2)
            )
        ) AS TotalEncounterCost,


        AVG(
            CAST(
                e.BaseEncounterCost
                AS DECIMAL(18,2)
            )
        ) AS AverageEncounterCost


    FROM fact.FactEncounter AS e

    WHERE
        e.DateKey IS NOT NULL
        AND e.FacilityKey IS NOT NULL

    GROUP BY
        e.DateKey,
        e.FacilityKey
),


-- ============================================================
-- 7. Calculate daily readmissions
-- ============================================================

ReadmissionDaily AS
(
    SELECT
        e.DateKey,
        e.FacilityKey,

        COUNT_BIG(*)
            AS ReadmissionCount

    FROM fact.FactReadmission AS r

    INNER JOIN fact.FactEncounter AS e

        ON e.EncounterKey =
           r.ReadmissionEncounterKey

    WHERE
        r.IsReadmission30Day = 1

        AND e.DateKey IS NOT NULL
        AND e.FacilityKey IS NOT NULL

    GROUP BY
        e.DateKey,
        e.FacilityKey
),


-- ============================================================
-- 8. Combine metrics
-- ============================================================

FacilityDailySource AS
(
    SELECT
        e.DateKey,
        e.FacilityKey,

        e.PatientCount,
        e.EncounterCount,

        e.EmergencyVisitCount,
        e.InpatientVisitCount,
        e.OutpatientVisitCount,

        ISNULL(
            r.ReadmissionCount,
            CAST(0 AS BIGINT)
        ) AS ReadmissionCount,

        e.AverageLengthOfStay,

        e.TotalEncounterCost,
        e.AverageEncounterCost

    FROM EncounterDaily AS e

    LEFT JOIN ReadmissionDaily AS r

        ON r.DateKey =
           e.DateKey

        AND r.FacilityKey =
            e.FacilityKey
),


-- ============================================================
-- 9. Identify new Facility + Date combinations
-- ============================================================

NewFacilityDays AS
(
    SELECT
        source.DateKey,
        source.FacilityKey,

        source.PatientCount,
        source.EncounterCount,

        source.EmergencyVisitCount,
        source.InpatientVisitCount,
        source.OutpatientVisitCount,

        source.ReadmissionCount,

        source.AverageLengthOfStay,

        source.TotalEncounterCost,
        source.AverageEncounterCost,

        ROW_NUMBER() OVER
        (
            ORDER BY
                source.DateKey,
                source.FacilityKey
        ) AS RowNumber

    FROM FacilityDailySource AS source

    WHERE NOT EXISTS
    (
        SELECT 1

        FROM fact.FactFacilityDaily AS target

        WHERE
            target.DateKey =
                source.DateKey

            AND target.FacilityKey =
                source.FacilityKey
    )
)


-- ============================================================
-- 10. Insert new Facility-Day rows
-- ============================================================

INSERT INTO fact.FactFacilityDaily
(
    FacilityDailyKey,

    DateKey,
    FacilityKey,

    PatientCount,
    EncounterCount,

    EmergencyVisitCount,
    InpatientVisitCount,
    OutpatientVisitCount,

    ReadmissionCount,

    AverageLengthOfStay,

    TotalEncounterCost,
    AverageEncounterCost
)

SELECT
    CAST(@MaxFacilityDailyKey AS BIGINT)
    +
    CAST(RowNumber AS BIGINT),

    DateKey,
    FacilityKey,

    PatientCount,
    EncounterCount,

    EmergencyVisitCount,
    InpatientVisitCount,
    OutpatientVisitCount,

    ReadmissionCount,

    AverageLengthOfStay,

    TotalEncounterCost,
    AverageEncounterCost

FROM NewFacilityDays;



-- ============================================================
-- 11. Summary
-- ============================================================

SELECT
    COUNT_BIG(*) AS TotalFacilityDayRows,

    COUNT_BIG(DISTINCT FacilityKey)
        AS TotalFacilities,

    SUM(
        CAST(PatientCount AS BIGINT)
    ) AS TotalDailyPatientCount,

    SUM(
        CAST(EncounterCount AS BIGINT)
    ) AS TotalEncounters,

    SUM(
        CAST(EmergencyVisitCount AS BIGINT)
    ) AS EmergencyVisits,

    SUM(
        CAST(InpatientVisitCount AS BIGINT)
    ) AS InpatientVisits,

    SUM(
        CAST(OutpatientVisitCount AS BIGINT)
    ) AS OutpatientVisits,

    SUM(
        CAST(ReadmissionCount AS BIGINT)
    ) AS Readmissions,

    SUM(
        CAST(TotalEncounterCost AS DECIMAL(18,2))
    ) AS TotalEncounterCost

FROM fact.FactFacilityDaily;



-- ============================================================
-- 12. Validate grain
--
-- Expected result:
--   ZERO rows
--
-- There must be only one record for each:
--   DateKey + FacilityKey
-- ============================================================

SELECT
    DateKey,
    FacilityKey,

    COUNT_BIG(*) AS RowCounts

FROM fact.FactFacilityDaily

GROUP BY
    DateKey,
    FacilityKey

HAVING COUNT_BIG(*) > 1;